# 13 — Classical baselines (subject-level LOSO-CV)

The floor the deep models must beat. Logistic Regression, RBF-SVM and Random
Forest on the trial feature table, evaluated with **leave-one-subject-out**
cross-validation and **subject-level scoring** (trial probabilities averaged
per held-out subject). Logic lives in `src.baseline`.

Five feature sets are compared, including a **duration-only control**: if a
model can't beat predicting from trial length alone, it hasn't learned gait.
A label-permutation null gives an honest p-value, and subject-bootstrap gives
the AUC confidence interval.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display
from src import config as C, baseline as B

df = C.load_trial_table()
sets = B.feature_sets(df)
{k: len(v) for k, v in sets.items()}

## 1. Run LOSO-CV across models × feature sets

Set `run_permutation=False` for a fast pass; `True` runs the 1000-shuffle null
(a few minutes). Results are cached to `reports/baseline_summary.json` and
`outputs/metrics/baseline_auc_grid.csv`.

In [ ]:
summary = B.main(run_permutation=True)
display(Image(str(C.FIGURES / 'baseline_auc_grid.png')))

## 2. ROC by feature set (exposing the confound)

Compare `shape_clean` (honest gait signal) against `confounded_only` and
`duration_only`. The gap between `shape_clean` and `duration_only` is the
*real* discriminative signal in the handcrafted features.

In [ ]:
display(Image(str(C.FIGURES / 'baseline_roc.png')))

## 3. Headline numbers

In [ ]:
grid = pd.read_csv(C.METRICS / 'baseline_auc_grid.csv', index_col=0)
display(grid.round(3))
print('Best config:', json.dumps(summary['best'], indent=2))
if 'permutation' in summary:
    print('Permutation test:', json.dumps(summary['permutation'], indent=2))

## Interpretation

- Subject-level AUC for trial features lands in the **~0.6–0.68** range —
  weak. The `duration_only` control sits near 0.61, so much of the apparent
  signal is trial-length confound, not gait.
- This is the honest floor. It motivates two things the deep-learning route
  fixes: (1) **fixed-length windows remove the duration confound by
  construction**, and (2) a CNN can use spatial micro-Doppler structure the
  handcrafted summaries throw away.
- Report `shape_clean` as the headline classical number (confound-controlled),
  with `duration_only` as the reference floor, in the thesis.

Next: build the windowed `.npy` cache (Notebook 06) and train the CNN / ResNet
(Notebooks 08–09) on a GPU — those need the conda env with torch.